# Creating a Simple Neural Network - Iris Flower Classifier  
John Elder, Codemy: https://youtu.be/O9Jx93DAyw4?si=1Pmt5GnwJObynZA3 

### 1. Define Model


In [ ]:
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import pandas as pd

from collections import deque

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve, roc_curve, roc_auc_score, average_precision_score

RANDOM_SEED = 42
IMAGE_DIR = "../../results/images/ClassifierNeuralNet"
DATASET_DIR = "../../data/processed/supervised/"

torch.manual_seed(RANDOM_SEED)
os.makedirs(IMAGE_DIR, exist_ok=True)

In [ ]:
# create model class that inherits nn.Module
class Model(nn.Module):
    # input layer (4 features of iris flower)
    # --> hidden layer H1 (8 neurons)
    # --> H2 (9 neurons)
    # --> output (3 classes of iris flowers)
    
    def __init__(self, in_features=30, h1=32, h2=16, out_features=1, dropout=0.2):
        super().__init__() # instantiate nn.Module
        self.fc1 = nn.Linear(in_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, out_features)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.gelu(self.fc1(x)) # rectified linear unit activation function
        x = self.dropout(x)
        x = F.gelu(self.fc2(x))
        x = self.dropout(x)
        x = self.out(x)
        return x

In [ ]:
# create instance of model
model = Model()

### 2. Pull Raw Data from GitHub Iris Classifier Dataset


In [ ]:
train_df = pd.read_csv(f'{DATASET_DIR}/train_data.csv')
val_df = pd.read_csv(f'{DATASET_DIR}/val_data.csv')
test_df = pd.read_csv(f'{DATASET_DIR}/test_data.csv')

### 4. Split Preprocessed Dataset into Train (80%) / Test (20%)


In [ ]:
X_train = train_df.drop('fraud', axis=1)
y_train = train_df['fraud']

X_val = val_df.drop('fraud', axis=1)
y_val = val_df['fraud']

X_test = test_df.drop('fraud', axis=1)
y_test = test_df['fraud']

In [ ]:
# convert X features to float tensors
X_train = torch.FloatTensor(X_train.values)
X_val = torch.FloatTensor(X_val.values)
X_test = torch.FloatTensor(X_test.values)

# convert y labels to long (float) tensors
y_train = torch.FloatTensor(y_train.astype(float).values).unsqueeze(1)
y_val = torch.FloatTensor(y_val.astype(float).values).unsqueeze(1)
y_test = torch.FloatTensor(y_test.astype(float).values).unsqueeze(1)

### 5. Choose Loss Function and Optimizer


In [ ]:
num_nonfraud = (y_train == 0).sum()
num_fraud = (y_train == 1).sum()

pos_weight = torch.tensor([num_nonfraud / num_fraud], dtype=torch.float)

# set criterion of model to measure error
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# choose Adam optimizer, set learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)

### 6. Train the Model


In [ ]:
# train the model
epochs = 250
warmup_epochs = 15
early_epoch = 0

loss_window = deque(maxlen=5)
best_val_loss = float('inf')
best_model_state = None

losses = {"train": [], "val": []}


In [ ]:

for epoch in range(1, epochs + 1):
    model.train()
    # go forward and get prediction
    y_pred = model(X_train)

    # measure loss/error
    train_loss = criterion(y_pred, y_train)

    # keep track of losses
    losses["train"].append(train_loss.item())

    # backpropagation: take error rate of forward propagation + feed it back through network to fine-tune weights
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        y_val_pred = model(X_val)
        val_loss = criterion(y_val_pred, y_val)
    
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_model_state = model.state_dict()

    losses["val"].append(val_loss.item())

    # print every 10 epochs
    if epoch % 10 == 0:
        print(f'Epoch: {epoch}: Train-Loss: {train_loss.item():.4f}, Val-Loss: {val_loss.item():.4f}')

    if epoch >= max(loss_window.maxlen, warmup_epochs):
        mean = np.mean(loss_window)
        std = np.std(loss_window)

        improvement = mean - val_loss.item()
        if std == 0: std = 1e-8

        z_score = improvement / std
        if z_score < 1:
            print(f'Early stopping at epoch {epoch} with val loss {val_loss.item():.4f} (z-score: {z_score:.2f})')
            early_epoch = epoch
            break
    
    loss_window.append(val_loss.item())
    early_epoch = epoch

In [ ]:
model.load_state_dict(best_model_state)
# display on a graph
plt.plot(range(early_epoch), losses["train"], label='Train Loss')
plt.plot(range(early_epoch), losses["val"], label='Validation Loss')
plt.ylabel('Loss/Error')
plt.xlabel('Epoch')
plt.savefig(f'{IMAGE_DIR}/loss_curve.png')
plt.legend()

model.eval()

In [ ]:
correct = 0
test_results = {"prob": [], "pred": []}

with torch.no_grad():
    for i, data in enumerate(X_val):
        logit = model(data)
        prob = torch.sigmoid(logit).item()
        pred = 1 if prob >= 0.5 else 0

        print(f'{i+1}.) logit: {logit.item():.4f} \t prob: {prob:.4f} \t predicted: {pred} \t actual: {int(y_val[i].item())}')

        # correct or not
        if pred == int(y_val[i].item()):
            correct += 1

        test_results["prob"].append(prob)
        test_results["pred"].append(pred)

print(f'Correct: {correct} / {len(X_val)}')

In [ ]:
accuracy = accuracy_score(y_val, test_results["pred"])
precision = precision_score(y_val, test_results["pred"], zero_division=0)
recall = recall_score(y_val, test_results["pred"], zero_division=0)
f1 = f1_score(y_val, test_results["pred"], zero_division=0)

pr_auc = average_precision_score(y_val, test_results["prob"])
roc_auc = roc_auc_score(y_val, test_results["prob"])

# If doing multi-class (0, 1, 2), you must specify average='weighted' or 'macro'
print(f"Validation Accuracy:  {accuracy:.4f}")
print(f"Validation Precision: {precision:.4f}")
print(f"Validation Recall:    {recall:.4f}")
print(f"Validation F1 Score:  {f1:.4f}")

print(f"Validation ROC AUC:   {roc_auc:.4f}")
print(f"Validation PR AUC:    {pr_auc:.4f}")

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, test_results["prob"])

plt.plot(precision, recall, label=f'Decision Tree (PR AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/pr_curve_validation.svg')
plt.show()

In [ ]:
beta = 2    # Makes F2 score, puts more emphasis on recall
fbeta_scores = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall + 1e-8)

best_idx = fbeta_scores.argmax()
best_threshold = thresholds[best_idx]

torch.save({"model": model, "threshold": best_threshold}, "../../results/models/classifier_nn.pt" )

### 7. Evaluate the Model Using the Test Dataset


In [ ]:
# evaluate model on test dataset
with torch.no_grad():   # turn off back propagation
    y_eval = model.forward(X_test)
    loss = criterion(y_eval, y_test)

loss.item()

In [ ]:
correct = 0
test_results = {"prob": [], "pred": []}

with torch.no_grad():
    for i, data in enumerate(X_test):
        logit = model(data)
        prob = torch.sigmoid(logit).item()
        pred = 1 if prob >= best_threshold else 0

        print(f'{i+1}.) logit: {logit.item():.4f} \t prob: {prob:.4f} \t predicted: {pred} \t actual: {int(y_test[i].item())}')

        # correct or not
        if pred == int(y_test[i].item()):
            correct += 1

        test_results["prob"].append(prob)
        test_results["pred"].append(pred)

print(f'Correct: {correct} / {len(X_test)}')

In [ ]:
accuracy = accuracy_score(y_test, test_results["pred"])
precision = precision_score(y_test, test_results["pred"], zero_division=0)
recall = recall_score(y_test, test_results["pred"], zero_division=0)
f1 = f1_score(y_test, test_results["pred"], zero_division=0)

pr_auc = average_precision_score(y_test, test_results["prob"])
roc_auc = roc_auc_score(y_test, test_results["prob"])

# If doing multi-class (0, 1, 2), you must specify average='weighted' or 'macro'
print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")
print(f"Test F1 Score:  {f1:.4f}")

print(f"Test ROC AUC:   {roc_auc:.4f}")
print(f"Test PR AUC:    {pr_auc:.4f}")

In [ ]:
cf_matrix = confusion_matrix(y_test, test_results["pred"])

group_names = ["True Negatives", "False Positives", "False Negatives", "True Positives"]

group_counts = ["{0:0.0f}".format(value) for value in cf_matrix.flatten()]

labels = [f"{v1}\n{v2}" for v1, v2 in zip(group_names,group_counts)]
labels = np.array(labels).reshape(2,2)

matrix = sns.heatmap(cf_matrix, annot=labels, fmt='', cmap="viridis", cbar=False)
matrix.set_xlabel("Predicted Label")
matrix.set_ylabel("True Label")
matrix.set_xticklabels(["Negative", "Positive"])
matrix.set_yticklabels(["Negative", "Positive"])
plt.title('Confusion Matrix')
plt.savefig(f'{IMAGE_DIR}/Confusion_Matrix.svg')
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_results["prob"])

plt.plot(fpr, tpr, label=f"Classifier-NN (AUC = {roc_auc:.4f})")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/roc_curve_test.svg')
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, test_results["prob"])

plt.plot(recall, precision, label=f"Classifier-NN (AUC = {pr_auc:.4f})")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/pr_curve_test.svg')
plt.show()